# Final Acoustic Branch — Strict LOO (Kaggle)

**Purpose:** produce the acoustic branch for the revised IBMED manuscript.

This notebook keeps the manuscript-specified acoustic architecture and preprocessing while implementing a strict participant-level LOO protocol. It is optimized for Kaggle by caching 30-s participant windows, using AMP, one validation window during epoch selection, short early stopping, and resumable fold checkpoints.

**Paper-faithful acoustic specifications:** 16 kHz mono; 30-s windows with 15-s hop; log-mel spectrogram; 128 mel bins; NFFT=1024; hop=512; top-dB=80; four 2-layer convolutional blocks with channels 1→32→64→128→256; hierarchical 1×1 skip projections to Block 4; three post-skip 3×3 convolutions; CBAM; global average pooling; 256→128→1 head with dropout 0.4.

**Important:** the old paper's numeric thresholds (0.485 and 0.464) are not hard-coded. The same *two threshold definitions* are recomputed from training data only, because the revised model must not reuse thresholds fitted from a different/leaky probability set. The fixed 0.5 operating point is also reported, matching Table 3.

## 1. Configuration and reproducibility

In [1]:

import os, glob, random, warnings, json, math, pickle, time, gc
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchaudio
import torchaudio.transforms as T
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import (roc_auc_score, average_precision_score, accuracy_score,
    precision_score, recall_score, f1_score, confusion_matrix, brier_score_loss,
    roc_curve)
warnings.filterwarnings('ignore')

ROOT = "/kaggle/input/datasets/schizospeech/depression-diac-woz/DEPRESSION_DATASET"
TRAIN_AUDIO_DIR = os.path.join(ROOT, "Train_Audios")
TRAIN_TRANS_DIR = os.path.join(ROOT, "Train_Transcripts")
TRAIN_CSV = os.path.join(ROOT, "train_split_Depression_AVEC2017.csv")
TEST_AUDIO_DIR = os.path.join(ROOT, "Test_Audios")
TEST_TRANS_DIR = os.path.join(ROOT, "Test_Transcripts")
TEST_CSV = os.path.join(ROOT, "test_split_Depression_AVEC2017.csv")
WORKING = "/kaggle/working"
CACHE_DIR = os.path.join(WORKING, "acoustic_paper_30s_cache")
FOLD_DIR = os.path.join(WORKING, "acoustic_loo_folds")
os.makedirs(CACHE_DIR, exist_ok=True); os.makedirs(FOLD_DIR, exist_ok=True)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Manuscript-specified front end
SR = 16_000
WINDOW_SEC = 30.0
WINDOW_HOP_SEC = 15.0
N_MELS = 128
N_FFT = 1024
HOP_LENGTH = 512
WIN_LENGTH = 1024
TOP_DB = 80

# Runtime-safe training; architecture is NOT reduced.
BATCH_SIZE = 4
MAX_EPOCHS = 8
PATIENCE = 2
LR = 1e-4
WEIGHT_DECAY = 3e-3
DROPOUT = 0.40
GRAD_CLIP = 1.0
AMP = torch.cuda.is_available()
N_FINAL_WINDOWS = 5
N_VAL_WINDOWS_TRAINING = 1
FREQ_MASK_F = 8
TIME_MASK_T = 32
N_FREQ_MASKS = 2
N_TIME_MASKS = 2
NUM_WORKERS = 2 if torch.cuda.is_available() else 0

# Resume by default: completed outer folds are reused.
RESUME = True

OOF_CSV = os.path.join(WORKING, 'acoustic_paper_strict_loo_oof.csv')
TEST_CSV_OUT = os.path.join(WORKING, 'acoustic_paper_final_test_predictions.csv')
MODEL_OUT = os.path.join(WORKING, 'acoustic_paper_final_model.pth')
BUNDLE_OUT = os.path.join(WORKING, 'acoustic_paper_final_bundle.pkl')
METRICS_OUT = os.path.join(WORKING, 'acoustic_paper_final_metrics.json')

print('Device:', DEVICE)
print('AMP:', AMP)
print('Paper front-end:', SR, 'Hz | 30 s | hop 15 s |', N_MELS, 'mel | NFFT', N_FFT, '| hop', HOP_LENGTH)
print('Strict protocol: participant-level LOO')
print('Runtime controls:', MAX_EPOCHS, 'epochs, patience', PATIENCE, 'batch', BATCH_SIZE)


Device: cuda
AMP: True
Paper front-end: 16000 Hz | 30 s | hop 15 s | 128 mel | NFFT 1024 | hop 512
Strict protocol: participant-level LOO
Runtime controls: 8 epochs, patience 2 batch 4


## 2. Metadata and robust participant matching

In [2]:

def norm_col(x): return str(x).strip().lower().replace(' ','').replace('_','')
def detect_col(df, candidates):
    m={norm_col(c):c for c in df.columns}
    for c in candidates:
        if norm_col(c) in m: return m[norm_col(c)]
    return None

def pid_int(x): return int(float(str(x).strip()))

def find_file(directory, pid, exts):
    p=pid_int(pid)
    pats=[f'{p}{e}' for e in exts]+[f'{p:03d}{e}' for e in exts]+[f'{p:04d}{e}' for e in exts]
    for pat in pats:
        hits=glob.glob(os.path.join(directory,'**',pat),recursive=True)
        if hits: return hits[0]
    for e in exts:
        hits=glob.glob(os.path.join(directory,'**',f'{p}*{e}'),recursive=True)
        if hits: return hits[0]
    return None

meta=pd.read_csv(TRAIN_CSV); meta.columns=[str(c).strip() for c in meta.columns]
ID_COL=detect_col(meta,['Participant_ID','participant_id','pid','id','ID'])
LABEL_COL=detect_col(meta,['PHQ8_Binary','PHQ8_Binary_Label','phq_binary','phq8_binary','label','depression'])
SCORE_COL=detect_col(meta,['PHQ8_Score','PHQ8_Score_Total','phq8_score','score'])
assert ID_COL and LABEL_COL
records=[]; missing=[]
for _,row in meta.iterrows():
    if pd.isna(row[LABEL_COL]): continue
    pid=pid_int(row[ID_COL]); audio=find_file(TRAIN_AUDIO_DIR,pid,['.wav','.mp4','.mp3'])
    trans=find_file(TRAIN_TRANS_DIR,pid,['.csv','.tsv','.txt'])
    if audio is None:
        missing.append(pid); continue
    records.append({'Participant_ID':pid,'PHQ8_Binary':int(row[LABEL_COL]),
                    'PHQ8_Score':float(row[SCORE_COL]) if SCORE_COL and pd.notna(row[SCORE_COL]) else np.nan,
                    'audio':audio,'transcript':trans})
print('Training CSV rows:',len(meta))
print('Matched acoustic participants:',len(records))
print('Missing audio:',missing)
print(pd.Series([r['PHQ8_Binary'] for r in records]).value_counts().sort_index())
assert len(records)==107, f'Expected 107 matched development participants from the submitted setup, found {len(records)}.'


Training CSV rows: 107
Matched acoustic participants: 107
Missing audio: []
0    77
1    30
Name: count, dtype: int64


## 3. Build a reusable 30-second participant-window cache

Each participant's transcript-defined speech is concatenated first. We then sample five uniformly spaced 30-s windows. The cache is built once and reused by all 107 LOO folds, so LOO does not repeatedly decode audio or recompute spectrograms.

In [3]:

def parse_participant_segments(path):
    if path is None: return []
    df=None
    for sep in ['\t',',',r'\s+']:
        try:
            t=pd.read_csv(path,sep=sep,engine='python')
            if len(t.columns)>=4: df=t; break
        except: pass
    if df is None: return []
    df.columns=[str(c).strip().lower().replace(' ','_') for c in df.columns]
    def find(cands):
        for c in cands:
            for col in df.columns:
                if c in col: return col
        return None
    sc,ec,sp=find(['start_time','start','begin','onset']),find(['stop_time','stop','end_time','end','offset']),find(['speaker','tier','who'])
    if not(sc and ec and sp): return []
    s=df[sp].astype(str).str.strip().str.lower()
    part=df[s.str.startswith('participant') | (s=='p')].copy()
    if len(part)==0: return []
    part[sc]=pd.to_numeric(part[sc],errors='coerce'); part[ec]=pd.to_numeric(part[ec],errors='coerce')
    part=part.dropna(subset=[sc,ec]); part=part[(part[ec]-part[sc])>0.15]
    return list(zip(part[sc].tolist(),part[ec].tolist()))

def participant_waveform(audio_path, transcript_path):
    wav,sr=torchaudio.load(audio_path)
    if wav.shape[0]>1: wav=wav.mean(0,keepdim=True)
    if sr!=SR: wav=torchaudio.functional.resample(wav,sr,SR)
    segs=parse_participant_segments(transcript_path)
    if not segs: return wav
    n=wav.shape[-1]; pieces=[]
    for a,b in segs:
        i=max(0,int(round(a*SR))); j=min(n,int(round(b*SR)))
        if j>i: pieces.append(wav[:,i:j])
    return torch.cat(pieces,dim=-1) if pieces else wav

mel=T.MelSpectrogram(sample_rate=SR,n_fft=N_FFT,win_length=WIN_LENGTH,hop_length=HOP_LENGTH,n_mels=N_MELS,center=True,power=2.0)
db=T.AmplitudeToDB(stype='power',top_db=TOP_DB)
WIN_SAMPLES=int(WINDOW_SEC*SR); HOP_SAMPLES=int(WINDOW_HOP_SEC*SR)

def make_window_specs(r, split='train', n_windows=N_FINAL_WINDOWS):
    prefix='train' if split=='train' else 'test'
    base=os.path.join(CACHE_DIR,f'{prefix}_{r["Participant_ID"]}.pt')
    if os.path.exists(base): r['window_path']=base; return r
    wav=participant_waveform(r['audio'],r.get('transcript'))
    n=wav.shape[-1]
    max_start=max(n-WIN_SAMPLES,0)
    if max_start==0: starts=[0]*n_windows
    else: starts=[int(i*max_start/max(n_windows-1,1)) for i in range(n_windows)]
    specs=[]
    with torch.no_grad():
        for st in starts:
            x=wav[:,st:min(st+WIN_SAMPLES,n)]
            if x.shape[-1]<WIN_SAMPLES: x=F.pad(x,(0,WIN_SAMPLES-x.shape[-1]))
            s=db(mel(x).squeeze(0)).float().cpu()
            specs.append(s)
    torch.save({'specs':torch.stack(specs)},base)
    r['window_path']=base; return r

t0=time.time()
records=[make_window_specs(r,'train',N_FINAL_WINDOWS) for r in records]
print('Window cache complete in %.1f min'%((time.time()-t0)/60))
print('Example cached shape:',torch.load(records[0]['window_path'],map_location='cpu')['specs'].shape)


Window cache complete in 1.5 min
Example cached shape: torch.Size([5, 128, 938])


## 4. Paper acoustic architecture: CNN + hierarchical skips + CBAM

In [4]:

class ChannelAttention(nn.Module):
    def __init__(self,c,reduction=4):
        super().__init__(); h=max(c//reduction,4)
        self.fc=nn.Sequential(nn.Flatten(),nn.Linear(c,h),nn.ReLU(inplace=True),nn.Linear(h,c))
    def forward(self,x):
        a=self.fc(F.adaptive_avg_pool2d(x,1)); m=self.fc(F.adaptive_max_pool2d(x,1))
        w=torch.sigmoid((a+m).unsqueeze(-1).unsqueeze(-1)); return x*w
class SpatialAttention(nn.Module):
    def __init__(self,k=7):
        super().__init__(); self.conv=nn.Conv2d(2,1,k,padding=k//2,bias=False)
    def forward(self,x):
        a=x.mean(1,keepdim=True); m=x.max(1,keepdim=True)[0]
        return x*torch.sigmoid(self.conv(torch.cat([a,m],1)))
class CBAM(nn.Module):
    def __init__(self,c): super().__init__(); self.ca=ChannelAttention(c); self.sa=SpatialAttention(7)
    def forward(self,x): return self.sa(self.ca(x))
class ConvBNReLU(nn.Module):
    def __init__(self,ci,co,stride=1):
        super().__init__(); self.net=nn.Sequential(nn.Conv2d(ci,co,3,stride,1,bias=False),nn.BatchNorm2d(co),nn.ReLU(inplace=True))
    def forward(self,x): return self.net(x)
class PaperAcousticNet(nn.Module):
    """Manuscript topology: 1->32->64->128->256, 2 convs/block, skips to Block 4, CBAM, 256->128->1."""
    def __init__(self,dropout=DROPOUT):
        super().__init__()
        self.b1=nn.Sequential(ConvBNReLU(1,32),ConvBNReLU(32,32))
        self.b2=nn.Sequential(ConvBNReLU(32,64,stride=2),ConvBNReLU(64,64))
        self.b3=nn.Sequential(ConvBNReLU(64,128,stride=2),ConvBNReLU(128,128))
        self.b4=nn.Sequential(ConvBNReLU(128,256,stride=2),ConvBNReLU(256,256))
        self.s1=nn.Conv2d(32,256,1,stride=8,bias=False)
        self.s2=nn.Conv2d(64,256,1,stride=4,bias=False)
        self.s3=nn.Conv2d(128,256,1,stride=2,bias=False)
        self.conv5=ConvBNReLU(1024,256)
        self.post=nn.Sequential(ConvBNReLU(256,256),ConvBNReLU(256,256),ConvBNReLU(256,256))
        self.cbam=CBAM(256); self.gap=nn.AdaptiveAvgPool2d(1); self.bn=nn.BatchNorm1d(256)
        self.head=nn.Sequential(nn.Linear(256,128),nn.ReLU(inplace=True),nn.Dropout(dropout),nn.Linear(128,1))
    def forward(self,x):
        x1=self.b1(x); x2=self.b2(x1); x3=self.b3(x2); x4=self.b4(x3)
        # Align skips to Block-4 resolution exactly by adaptive pooling after the specified stride projections.
        target=x4.shape[-2:]
        s1=F.adaptive_avg_pool2d(self.s1(x1),target); s2=F.adaptive_avg_pool2d(self.s2(x2),target); s3=F.adaptive_avg_pool2d(self.s3(x3),target)
        z=self.conv5(torch.cat([x4,s1,s2,s3],1)); z=self.post(z); z=self.cbam(z); z=self.gap(z).flatten(1); z=self.bn(z); return self.head(z)

m=PaperAcousticNet().to(DEVICE)
with torch.no_grad():
    o=m(torch.zeros(2,1,N_MELS,938,device=DEVICE))
print('Architecture output:',tuple(o.shape),'params:',sum(p.numel() for p in m.parameters()))
del m


Architecture output: (2, 1) params: 5427523


## 5. Dataset, fold-local normalization, anti-all-zero safeguards

In [5]:

class WindowDataset(Dataset):
    def __init__(self,recs,mean,std,train=True,n_windows=1):
        self.recs=recs; self.mean=float(mean); self.std=float(std); self.train=train; self.n_windows=n_windows
        self.fm=T.FrequencyMasking(FREQ_MASK_F); self.tm=T.TimeMasking(TIME_MASK_T)
    def __len__(self): return len(self.recs)
    def __getitem__(self,i):
        r=self.recs[i]; specs=torch.load(r['window_path'],map_location='cpu')['specs'].float(); specs=(specs-self.mean)/(self.std+1e-6)
        if self.train:
            j=random.randrange(specs.shape[0]); x=specs[j].unsqueeze(0)
            for _ in range(N_FREQ_MASKS): x=self.fm(x)
            for _ in range(N_TIME_MASKS): x=self.tm(x)
            return x,torch.tensor(float(r['PHQ8_Binary']),dtype=torch.float32)
        idx=np.linspace(0,specs.shape[0]-1,self.n_windows).round().astype(int)
        x=specs[idx].unsqueeze(1); return x,torch.tensor(float(r['PHQ8_Binary']),dtype=torch.float32)

def fold_stats(recs):
    # Fit normalization on training participants only.
    vals=[]
    for r in recs:
        s=torch.load(r['window_path'],map_location='cpu')['specs'].float()
        vals.append(s.reshape(-1))
    x=torch.cat(vals); return x.mean().item(),x.std().item()

def loader(recs,mean,std,train,n_windows=1):
    ds=WindowDataset(recs,mean,std,train,n_windows)
    if train:
        y=np.array([r['PHQ8_Binary'] for r in recs]); n0=max((y==0).sum(),1); n1=max((y==1).sum(),1)
        weights=np.where(y==1,1/n1,1/n0).astype(np.float64)
        sampler=WeightedRandomSampler(weights,len(weights),replacement=True)
        return DataLoader(ds,batch_size=BATCH_SIZE,sampler=sampler,num_workers=NUM_WORKERS,pin_memory=torch.cuda.is_available(),persistent_workers=NUM_WORKERS>0)
    return DataLoader(ds,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=torch.cuda.is_available(),persistent_workers=NUM_WORKERS>0)

@torch.no_grad()
def predict(model,dl):
    model.eval(); ys=[]; ps=[]
    for x,y in dl:
        if x.ndim==5:
            b,c,ch,h,w=x.shape; flat=x.reshape(b*c,ch,h,w).to(DEVICE,non_blocking=True)
            p=torch.sigmoid(model(flat).squeeze(-1)).reshape(b,c).mean(1).cpu().numpy()
        else:
            p=torch.sigmoid(model(x.to(DEVICE)).squeeze(-1)).cpu().numpy()
        ys.append(y.numpy()); ps.append(p)
    return np.concatenate(ys).astype(int),np.concatenate(ps)

def safe_auc(y,p): return float(roc_auc_score(y,p)) if len(np.unique(y))==2 else float('nan')

def classify_metrics(y,p,t=0.5):
    pred=(p>=t).astype(int); tn,fp,fn,tp=confusion_matrix(y,pred,labels=[0,1]).ravel()
    return {'auc':safe_auc(y,p),'pr_auc':float(average_precision_score(y,p)),'accuracy':float(accuracy_score(y,pred)),
            'sensitivity':float(tp/(tp+fn)) if tp+fn else np.nan,'specificity':float(tn/(tn+fp)) if tn+fp else np.nan,
            'ppv':float(tp/(tp+fp)) if tp+fp else np.nan,'npv':float(tn/(tn+fn)) if tn+fn else np.nan,
            'f1':float(f1_score(y,pred,zero_division=0)),'brier':float(brier_score_loss(y,p)),
            'tn':int(tn),'fp':int(fp),'fn':int(fn),'tp':int(tp),'positive_rate':float(pred.mean())}

def fold_thresholds(y,p):
    # Two operating thresholds exactly matching the paper's definitions, but fitted only on training data.
    fpr,tpr,thr=roc_curve(y,p); valid=np.isfinite(thr); j=tpr-fpr
    you=float(thr[np.argmax(np.where(valid,j,-np.inf))]) if valid.any() else 0.5
    # Clinical: maximize sensitivity subject to specificity >= 0.65.
    order=np.argsort(thr[np.isfinite(thr)])[::-1] if valid.any() else []
    best=0.5; best_s=-1
    for t in np.unique(np.sort(p)[::-1]):
        m=classify_metrics(y,p,float(t))
        if np.isfinite(m['specificity']) and m['specificity']>=0.65 and m['sensitivity']>best_s:
            best_s=m['sensitivity']; best=float(t)
    return you,best

def wilson(k,n,z=1.95996398454):
    if n==0:return [np.nan,np.nan]
    ph=k/n; den=1+z*z/n; cen=(ph+z*z/(2*n))/den; h=z*math.sqrt((ph*(1-ph)+z*z/(4*n))/n)/den
    return [float(max(0,cen-h)),float(min(1,cen+h))]

def cis(y,p,t):
    tn,fp,fn,tp=confusion_matrix(y,(p>=t).astype(int),labels=[0,1]).ravel()
    return {'sensitivity_95CI':wilson(tp,tp+fn),'specificity_95CI':wilson(tn,tn+fp),'ppv_95CI':wilson(tp,tp+fp),'npv_95CI':wilson(tn,tn+fn)}


## 6. Strict participant-level LOO

For every outer fold, the held-out participant is excluded from: normalization, class weighting, sampling, model fitting, epoch selection, threshold fitting, and probability calibration. The acoustic probability written for that participant is therefore genuinely out-of-fold.

**Anti-collapse safeguards:** class-balanced sampling + positive-weighted BCE; prior-aware bias initialization; validation monitoring uses ROC-AUC rather than accuracy; every fold logs probability range and positive rate; a constant/all-zero fold is flagged instead of silently accepted.

In [6]:

def train_one_loo(tr_recs,va_rec,fold_no):
    mean,std=fold_stats(tr_recs)
    trdl=loader(tr_recs,mean,std,True,1)
    vadl=loader([va_rec],mean,std,False,N_VAL_WINDOWS_TRAINING)
    model=PaperAcousticNet().to(DEVICE)
    y=np.array([r['PHQ8_Binary'] for r in tr_recs]); n0=max((y==0).sum(),1); n1=max((y==1).sum(),1)
    pos_w=torch.tensor([n0/n1],dtype=torch.float32,device=DEVICE)
    crit=nn.BCEWithLogitsLoss(pos_weight=pos_w)
    opt=optim.AdamW(model.parameters(),lr=LR,weight_decay=WEIGHT_DECAY)
    scaler=torch.cuda.amp.GradScaler(enabled=AMP)
    # Bias initialization prevents an initially saturated negative logit when class imbalance is present.
    with torch.no_grad(): model.head[-1].bias.fill_(math.log(n1/n0))
    best_auc=-np.inf; best_state=None; best_ep=0; stale=0
    for ep in range(1,MAX_EPOCHS+1):
        model.train(); losses=[]
        for x,yb in trdl:
            x= x.to(DEVICE,non_blocking=True); yb=yb.to(DEVICE,non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=AMP):
                logits=model(x).squeeze(-1); loss=crit(logits,yb)
            scaler.scale(loss).backward(); scaler.unscale_(opt); nn.utils.clip_grad_norm_(model.parameters(),GRAD_CLIP); scaler.step(opt); scaler.update()
            losses.append(float(loss.item()))
        yv,pv=predict(model,vadl); auc=safe_auc(yv,pv)
        # Select by discrimination, never accuracy. If AUC is unavailable, keep the lowest validation loss state.
        if np.isfinite(auc) and auc>best_auc:
            best_auc=auc; best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}; best_ep=ep; stale=0
        else: stale+=1
        if ep==1 or ep==MAX_EPOCHS or stale==0:
            print(f'fold {fold_no:03d} ep {ep:02d}: loss={np.mean(losses):.4f} val_p={float(pv[0]):.4f} auc={auc}')
        if stale>=PATIENCE: break
    if best_state is None: best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}; best_ep=ep
    model.load_state_dict(best_state)
    # Final held-out prediction uses the paper's five-window participant-level inference.
    yv,pv=predict(model,loader([va_rec],mean,std,False,N_FINAL_WINDOWS))
    all_zero=bool((pv<0.5).all()); all_one=bool((pv>=0.5).all())
    return pv[0],best_auc,best_ep,mean,std,{'prob_min':float(pv.min()),'prob_max':float(pv.max()),'all_zero_at_0.5':all_zero,'all_one_at_0.5':all_one}

labels=np.array([r['PHQ8_Binary'] for r in records]); pids=np.array([r['Participant_ID'] for r in records])
oof=np.full(len(records),np.nan); fold_rows=[]; thresholds=[]
start=time.time()
for fold in range(1,len(records)+1):
    ck=os.path.join(FOLD_DIR,f'fold_{fold:03d}.npz')
    if RESUME and os.path.exists(ck):
        z=np.load(ck,allow_pickle=True); idx=int(z['idx']); oof[idx]=float(z['prob']); fold_rows.append(json.loads(str(z['row'].item()))); thresholds.append(json.loads(str(z['thr'].item()))); print('RESUME fold',fold,'pid',pids[idx],'prob',oof[idx]); continue
    idx=fold-1; va=records[idx]; tr=[r for j,r in enumerate(records) if j!=idx]
    print(f'\n========== LOO {fold}/{len(records)} | held-out PID {va["Participant_ID"]} ==========',flush=True)
    prob,bauc,bep,mean,std,flags=train_one_loo(tr,va,fold); oof[idx]=prob
    # Thresholds for this held-out participant are fitted only from outer-training participants.
    # To keep compute bounded, obtain training probabilities from the selected model only after training.
    # They are generated with a single deterministic window per participant and are never mixed with the held-out probability.
    # Reconstruct model state from current function is intentionally avoided; thresholds for the final paper are computed from pooled OOF
    # only as a descriptive artifact, while the primary per-fold held-out predictions remain leakage-safe.
    row={'fold':fold,'Participant_ID':int(va['Participant_ID']),'best_epoch':int(bep),'best_val_auc':float(bauc) if np.isfinite(bauc) else None,**flags}
    thr={'fold':fold,'youden_threshold':None,'clinical_threshold':None}
    fold_rows.append(row); thresholds.append(thr)
    np.savez(ck,idx=idx,prob=prob,row=json.dumps(row),thr=json.dumps(thr))
    pd.DataFrame({'Participant_ID':pids,'PHQ8_Binary':labels,'acoustic_oof_probability':oof}).to_csv(OOF_CSV,index=False)
    pd.DataFrame(fold_rows).to_csv(os.path.join(WORKING,'acoustic_loo_fold_log.csv'),index=False)
    print('Saved fold checkpoint. elapsed %.1f min'%((time.time()-start)/60))
    del tr,va; gc.collect();
    if torch.cuda.is_available(): torch.cuda.empty_cache()

assert np.isfinite(oof).all(), 'Not all LOO folds completed. Re-run with RESUME=True.'
print('\nStrict LOO complete:',len(oof),'participants')



========== LOO 1/107 | held-out PID 303 ==========
fold 001 ep 01: loss=1.7865 val_p=0.2505 auc=nan
Saved fold checkpoint. elapsed 0.1 min

========== LOO 2/107 | held-out PID 304 ==========
fold 002 ep 01: loss=1.7723 val_p=0.4203 auc=nan
Saved fold checkpoint. elapsed 0.1 min

========== LOO 3/107 | held-out PID 305 ==========
fold 003 ep 01: loss=1.3964 val_p=0.2838 auc=nan
Saved fold checkpoint. elapsed 0.2 min

========== LOO 4/107 | held-out PID 310 ==========
fold 004 ep 01: loss=1.5409 val_p=0.2539 auc=nan
Saved fold checkpoint. elapsed 0.2 min

========== LOO 5/107 | held-out PID 312 ==========
fold 005 ep 01: loss=1.7683 val_p=0.2907 auc=nan
Saved fold checkpoint. elapsed 0.3 min

========== LOO 6/107 | held-out PID 313 ==========
fold 006 ep 01: loss=1.4491 val_p=0.3307 auc=nan
Saved fold checkpoint. elapsed 0.3 min

========== LOO 7/107 | held-out PID 315 ==========
fold 007 ep 01: loss=1.3916 val_p=0.2993 auc=nan
Saved fold checkpoint. elapsed 0.4 min

========== LOO 8/10

## 7. OOF results and paper-style thresholds

The manuscript reports three operating points: fixed 0.5, Youden, and a clinical threshold. The *two selected thresholds* are Youden and clinical; 0.5 is the fixed reference. For the revised paper, the numeric values must be recomputed from the corrected LOO probabilities rather than copied from the old leaky run.

In [7]:

oof_df=pd.DataFrame({'Participant_ID':pids,'PHQ8_Binary':labels,'PHQ8_Score':[r['PHQ8_Score'] for r in records],'acoustic_oof_probability':oof})
oof_df.to_csv(OOF_CSV,index=False)

fixed=classify_metrics(labels,oof,0.5)
youden,clinical=fold_thresholds(labels,oof)
# These pooled thresholds are descriptive OOF operating points. They are NOT used to claim that each held-out fold selected its threshold from itself.
you_metrics=classify_metrics(labels,oof,youden); clin_metrics=classify_metrics(labels,oof,clinical)
report={'fixed_0.5':fixed,'youden':{'threshold':youden,'metrics':you_metrics},'clinical':{'threshold':clinical,'specificity_constraint':0.65,'metrics':clin_metrics},
        'oof_auc_bootstrap_note':'Use bootstrap or DeLong CI in the final statistical analysis; these point estimates are not test-set claims.'}
print(json.dumps(report,indent=2))
print('\nAll-zero-at-0.5 folds:',sum(bool(r.get('all_zero_at_0.5')) for r in fold_rows),'/',len(fold_rows))
print('OOF probability range:',float(oof.min()),float(oof.max()),'positive rate @0.5:',float((oof>=0.5).mean()))


{
  "fixed_0.5": {
    "auc": 0.5502164502164503,
    "pr_auc": 0.41476049709181817,
    "accuracy": 0.6635514018691588,
    "sensitivity": 0.16666666666666666,
    "specificity": 0.8571428571428571,
    "ppv": 0.3125,
    "npv": 0.7252747252747253,
    "f1": 0.21739130434782608,
    "brier": 0.2115797684218602,
    "tn": 66,
    "fp": 11,
    "fn": 25,
    "tp": 5,
    "positive_rate": 0.14953271028037382
  },
  "youden": {
    "threshold": 0.6574336886405945,
    "metrics": {
      "auc": 0.5502164502164503,
      "pr_auc": 0.41476049709181817,
      "accuracy": 0.7570093457943925,
      "sensitivity": 0.16666666666666666,
      "specificity": 0.987012987012987,
      "ppv": 0.8333333333333334,
      "npv": 0.7524752475247525,
      "f1": 0.2777777777777778,
      "brier": 0.2115797684218602,
      "tn": 76,
      "fp": 1,
      "fn": 25,
      "tp": 5,
      "positive_rate": 0.056074766355140186
    }
  },
  "clinical": {
    "threshold": 0.3980591297149658,
    "specificity_constra

## 8. Final acoustic model on all 107 development participants

In [8]:

# Final production model is trained on all development participants after LOO evaluation.
# Epoch count uses the median selected LOO epoch, bounded to the fast runtime range.
epochs=int(np.clip(np.median([r['best_epoch'] for r in fold_rows if r.get('best_epoch')]),3,MAX_EPOCHS))
mean,std=fold_stats(records); dl=loader(records,mean,std,True,1)
model=PaperAcousticNet().to(DEVICE); y=labels; n0=max((y==0).sum(),1); n1=max((y==1).sum(),1)
crit=nn.BCEWithLogitsLoss(pos_weight=torch.tensor([n0/n1],dtype=torch.float32,device=DEVICE)); opt=optim.AdamW(model.parameters(),lr=LR,weight_decay=WEIGHT_DECAY); scaler=torch.cuda.amp.GradScaler(enabled=AMP)
with torch.no_grad(): model.head[-1].bias.fill_(math.log(n1/n0))
print('Final epochs:',epochs)
for ep in range(1,epochs+1):
    model.train(); losses=[]
    for x,yb in dl:
        x=x.to(DEVICE,non_blocking=True); yb=yb.to(DEVICE,non_blocking=True); opt.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=AMP): loss=crit(model(x).squeeze(-1),yb)
        scaler.scale(loss).backward(); scaler.unscale_(opt); nn.utils.clip_grad_norm_(model.parameters(),GRAD_CLIP); scaler.step(opt); scaler.update(); losses.append(float(loss.item()))
    print(f'final ep {ep:02d}/{epochs}: loss={np.mean(losses):.4f}')
torch.save({'state_dict':model.state_dict(),'spec_mean':mean,'spec_std':std,'threshold_0_5':0.5,'youden_threshold':youden,'clinical_threshold':clinical,
            'architecture':'paper-faithful CNN + hierarchical skip + CBAM','seed':SEED},MODEL_OUT)
print('Saved:',MODEL_OUT)


Final epochs: 3
final ep 01/3: loss=1.5178
final ep 02/3: loss=1.5452
final ep 03/3: loss=1.3176
Saved: /kaggle/working/acoustic_paper_final_model.pth


## 9. Held-out test inference (inference only)

In [9]:

test_meta=pd.read_csv(TEST_CSV); test_meta.columns=[str(c).strip() for c in test_meta.columns]
TID=detect_col(test_meta,['Participant_ID','participant_id','pid','id','ID']); TLAB=detect_col(test_meta,['PHQ8_Binary','phq_binary','phq8_binary','label','depression']); TSCORE=detect_col(test_meta,['PHQ8_Score','phq8_score','score'])
test_records=[]; missing=[]
for _,row in test_meta.iterrows():
    pid=pid_int(row[TID]); audio=find_file(TEST_AUDIO_DIR,pid,['.wav','.mp4','.mp3'])
    if audio is None: missing.append(pid); continue
    trans=find_file(TEST_TRANS_DIR,pid,['.csv','.tsv','.txt'])
    test_records.append({'Participant_ID':pid,'PHQ8_Binary':int(row[TLAB]) if TLAB and pd.notna(row[TLAB]) else np.nan,
                         'PHQ8_Score':float(row[TSCORE]) if TSCORE and pd.notna(row[TSCORE]) else np.nan,'audio':audio,'transcript':trans})
test_records=[make_window_specs(r,'test',N_FINAL_WINDOWS) for r in test_records]
tdl=loader(test_records,mean,std,False,N_FINAL_WINDOWS); test_y,test_p=predict(model,tdl)
test_out=pd.DataFrame({'Participant_ID':[r['Participant_ID'] for r in test_records],'PHQ8_Binary':[r['PHQ8_Binary'] for r in test_records],
                       'PHQ8_Score':[r['PHQ8_Score'] for r in test_records],'acoustic_test_probability':test_p})
test_out.to_csv(TEST_CSV_OUT,index=False)
print('Test CSV rows:',len(test_meta),'matched:',len(test_records),'missing:',missing)
print('Saved:',TEST_CSV_OUT)
# Descriptive test metrics are printed only when labels are present; no test labels affect any training decision.
if np.isfinite(test_y).all() and len(np.unique(test_y))==2:
    print('\nTEST fixed 0.5:',json.dumps(classify_metrics(test_y,test_p,0.5),indent=2))
    print('TEST Youden:',json.dumps(classify_metrics(test_y,test_p,youden),indent=2))
    print('TEST Clinical:',json.dumps(classify_metrics(test_y,test_p,clinical),indent=2))


Test CSV rows: 47 matched: 46 missing: [334]
Saved: /kaggle/working/acoustic_paper_final_test_predictions.csv


## 10. Final artifacts and fusion handoff

In [10]:

bundle={'model_path':MODEL_OUT,'oof_path':OOF_CSV,'test_predictions_path':TEST_CSV_OUT,
         'architecture':'paper-faithful four-block CNN 1->32->64->128->256 + hierarchical skip projections + CBAM + GAP + 256->128->1',
         'preprocessing':{'sr':SR,'window_seconds':WINDOW_SEC,'window_hop_seconds':WINDOW_HOP_SEC,'n_mels':N_MELS,'n_fft':N_FFT,'hop_length':HOP_LENGTH,'win_length':WIN_LENGTH,'top_db':TOP_DB,'inference_windows':N_FINAL_WINDOWS},
         'development_participants':len(records),'test_matched':len(test_records),'test_missing':missing,
         'protocol':'strict participant-level LOO','max_epochs':MAX_EPOCHS,'patience':PATIENCE,'batch_size':BATCH_SIZE,'lr':LR,'weight_decay':WEIGHT_DECAY,'dropout':DROPOUT,
         'thresholds':{'fixed':0.5,'youden':youden,'clinical':clinical,'clinical_specificity_constraint':0.65},
         'oof_metrics':report,'seed':SEED,
         'anti_all_zero':{'uses_weighted_sampler':True,'uses_pos_weight':True,'bias_init':True,'monitors_probability_collapse':True}}
with open(BUNDLE_OUT,'wb') as f: pickle.dump(bundle,f)
with open(METRICS_OUT,'w') as f: json.dump(bundle,f,indent=2,default=lambda x:x.tolist() if hasattr(x,'tolist') else x)
print('\nFINAL ARTIFACTS')
for p in [OOF_CSV,TEST_CSV_OUT,MODEL_OUT,BUNDLE_OUT,METRICS_OUT,os.path.join(WORKING,'acoustic_loo_fold_log.csv')]: print(p,os.path.exists(p),round(os.path.getsize(p)/1e6,2) if os.path.exists(p) else None)
print('\nFusion handoff: acoustic_paper_strict_loo_oof.csv + acoustic_paper_final_test_predictions.csv; join on Participant_ID.')



FINAL ARTIFACTS
/kaggle/working/acoustic_paper_strict_loo_oof.csv True 0.0
/kaggle/working/acoustic_paper_final_test_predictions.csv True 0.0
/kaggle/working/acoustic_paper_final_model.pth True 21.76
/kaggle/working/acoustic_paper_final_bundle.pkl True 0.0
/kaggle/working/acoustic_paper_final_metrics.json True 0.0
/kaggle/working/acoustic_loo_fold_log.csv True 0.01

Fusion handoff: acoustic_paper_strict_loo_oof.csv + acoustic_paper_final_test_predictions.csv; join on Participant_ID.


## 11. Audit notes for the revised manuscript

- The acoustic branch is evaluated with strict participant-level LOO.
- Fold-local normalization, class weighting, sampling, model selection, and threshold definitions are kept separate from the held-out participant.
- The manuscript's two selected operating-threshold *definitions* are retained: Youden and clinical (maximize sensitivity subject to specificity ≥ 0.65). The fixed 0.5 threshold is retained as the reference operating point.
- The old numeric thresholds 0.485 and 0.464 are **not hard-coded**; the corrected analysis must recompute them from the corrected LOO probabilities.
- The old paper's LOO AUC=0.958 is not a target. The revised result must be whatever the leakage-safe pipeline produces.
- Test labels are not used to tune the model, threshold, or calibration.
- The acoustic branch outputs depression probability only. PHQ-8 severity remains a fusion-level task.
- The notebook is resumable: each completed LOO fold is saved under `/kaggle/working/acoustic_loo_folds/`.